# SpecEdge: Adaptive Speculative Decoding & Quantization Dynamics on Edge SLMs
**Author:** Benedict Baah  
**Project Repository:** `edge-slm-speculative-decoding`  

This turnkey notebook demonstrates **SpecEdge**, an empirical framework evaluating speculative decoding under low-bit quantization (INT4-NF4 / INT8 / FP16) and introducing **Entropy-Gated Adaptive Speculation (EG-Spec)** on a free Google Colab T4/A100 GPU.

In [ ]:
# 1. Environment & Hardware Verification
!nvidia-smi
!pip install -q torch transformers accelerate bitsandbytes scipy numpy pandas matplotlib seaborn tabulate psutil

In [ ]:
# 2. Clone or Import SpecEdge Codebase
import os, sys, time, torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()} | Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 3. Load Target (INT4) and Draft (FP16) Models
We use the ultra-compact **SmolLM** family:
* **Target Model**: `HuggingFaceTB/SmolLM-1.7B-Instruct` quantized to **INT4 (NF4)**
* **Draft Model**: `HuggingFaceTB/SmolLM-135M-Instruct` in **FP16**

In [ ]:
target_name = "HuggingFaceTB/SmolLM-1.7B-Instruct"
draft_name = "HuggingFaceTB/SmolLM-135M-Instruct"

bnb_config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading Target Model in INT4-NF4...")
tokenizer = AutoTokenizer.from_pretrained(target_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

target_model = AutoModelForCausalLM.from_pretrained(
    target_name,
    quantization_config=bnb_config_4bit,
    device_map="auto"
)

print("Loading Draft Model in FP16...")
draft_model = AutoModelForCausalLM.from_pretrained(
    draft_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Models loaded successfully!")

## 4. Run SpecEdge Interactive Generation Demo
Observe real-time speculative execution with step-by-step acceptance tracing.

In [ ]:
from src.core.speculative_engine import SpeculativeEngine
from src.core.adaptive_policy import EntropyGatedPolicy

engine = SpeculativeEngine(
    target_model=target_model,
    draft_model=draft_model,
    tokenizer=tokenizer,
    device="cuda"
)

prompt = "Question: A bookstore sold 25% of its 120 books on Monday. On Tuesday, they received 40 books. How many books remain? Explain step by step."
inputs = tokenizer(prompt, return_tensors="pt")["input_ids"]

# 1. Baseline Autoregressive
print("Running Autoregressive Baseline...")
out_ar = engine.generate_autoregressive(inputs, max_new_tokens=60)

# 2. SpecEdge with Entropy-Gated Lookahead
print("Running SpecEdge (EG-Spec Adaptive)...")
policy = EntropyGatedPolicy(entropy_threshold=1.45, max_gamma=5, enabled=True)
out_spec = engine.generate_speculative(inputs, gamma=5, max_new_tokens=60, adaptive_policy=policy)

speedup = out_ar.profile.ms_per_token / max(out_spec.profile.ms_per_token, 1e-5)
print(f"\n=== RESULTS ===")
print(f"AR Latency: {out_ar.profile.ms_per_token:.2f} ms/token ({out_ar.profile.tokens_per_sec:.1f} tok/s)")
print(f"SpecEdge Latency: {out_spec.profile.ms_per_token:.2f} ms/token ({out_spec.profile.tokens_per_sec:.1f} tok/s)")
print(f"Wall-Clock Speedup: {speedup:.2f}x")
print(f"Mean Acceptance Rate: {out_spec.profile.acceptance_rate * 100:.1f}%")
print(f"Generated Text:\n{out_spec.text}")

## 5. Automated Benchmark Suite & Statistical Significance
Sweeps over Reasoning (GSM8K), Code (HumanEval), and Summarization (CNN/DM).

In [ ]:
from src.evaluation.benchmark_runner import BenchmarkSuite
from src.evaluation.statistical_analysis import StatisticalAnalyzer
from src.utils.visualizer import PublicationVisualizer

suite = BenchmarkSuite(engine=engine)
df_results = suite.run_evaluation_suite(gamma=5, max_new_tokens=80)
print(df_results[["prompt_id", "category", "ar_tokens_per_sec", "static_speedup", "eg_speedup", "eg_acc_rate"]])

# Statistical Tests
stat_dict = {}
for cat in df_results["category"].unique():
    sub = df_results[df_results["category"] == cat]
    stat_dict[cat] = StatisticalAnalyzer.compare_paired_methods(
        sub["ar_ms_per_token"].tolist(),
        sub["eg_ms_per_token"].tolist(),
        metric_name=cat
    )

print("\nStatistical Significance Analysis:")
print(StatisticalAnalyzer.format_latex_table(stat_dict))

# Generate Figures
vis = PublicationVisualizer(output_dir="./figures")
vis.plot_speedup_by_category(df_results)
print("Figures generated successfully in ./figures/")